# 🕵️ ISOM 260: Job Hunter — A Sneak Peek at Agents

**For the impatient.** | Suffolk University | Prof. Hasan Arslan

---

You asked for agents. Officially, agents start in **Session 4** (Wed Sep 30). But if you can't wait — and some of you literally wrote *"automated job applications"* on the whiteboard in week 1 — here's a taste.

**What you'll build in ~15 minutes:** a *proto*-agent that
1. 🔧 uses a **tool** — pulls real, live job listings from a public jobs API
2. 🧠 uses a **brain** — Gemini ranks them against *your* profile and explains the fit
3. ✍️ **acts** — drafts a tailored outreach note for the top match

**Requirements:** your API key from Wednesday's class in Colab Secrets (`GOOGLE_API_KEY`). No key yet? That's Wednesday — this notebook will be waiting.

> ⚖️ **Why not LinkedIn?** LinkedIn has no public job-search API, and scraping it violates their Terms of Service. Real builders use legitimate APIs — today's is [RemoteOK](https://remoteok.com), which publishes its remote-job listings as free public JSON (with a link back to them — which we include). Same skill, zero legal problems. Remember this instinct: *check the API terms before you build.*


In [ ]:
# ── Setup (same as Wednesday) ─────────────────────────────────────
!pip install -q google-genai

In [ ]:
# ── Load your API key from Colab Secrets ──────────────────────────
from google.colab import userdata
from google import genai

GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")   # 🔑 Secrets panel, from Wednesday
client = genai.Client(api_key=GOOGLE_API_KEY)
print("✅ Ready to hunt.")

## 👤 Step 1 — Who's hunting?

Edit this honestly — the ranking is only as good as the profile. (Notice: this is a plain Python dictionary. Structured data in, better AI out — a theme for the whole course.)


In [ ]:
# ── YOUR profile — edit me! ───────────────────────────────────────
my_profile = {
    "name": "Alex",
    "degree": "Business major, ISOM concentration, graduating May 2028",
    "skills": ["Excel", "market research", "social media", "Python (beginner)", "AI tools"],
    "interested_in": "marketing",        # <- the job search keyword: try "sales", "finance", "data", "customer success"...
    "wants": "a remote internship or junior role where I can use AI tools day-to-day",
    "dealbreakers": "no 100% cold-calling roles",
}
print(f"🎯 Hunting for '{my_profile['interested_in']}' roles for {my_profile['name']}...")

## 🔧 Step 2 — The tool: fetch real jobs

This function is a **tool**: a capability your AI brain can use but doesn't have on its own (Gemini can't browse job boards — but your code can, and then hands over the results). In Session 4, the agent will decide *for itself* when to call tools like this one.


In [ ]:
# ── TOOL: pull ~100 live listings from the RemoteOK public API ────
import urllib.request, json, re

def fetch_jobs(keyword, limit=8):
    """Fetch live remote jobs, filter by keyword, return a compact list."""
    req = urllib.request.Request("https://remoteok.com/api",
                                 headers={"User-Agent": "ISOM260-student-project"})
    raw = json.load(urllib.request.urlopen(req, timeout=25))
    pool = [j for j in raw if isinstance(j, dict) and j.get("position")]

    # OUR code narrows the pool — tools fetch, code filters, the model reasons
    kw = keyword.lower()
    hits = [j for j in pool
            if kw in (j.get("position","") + " " + " ".join(j.get("tags",[]))
                      + " " + j.get("description","")).lower()]

    jobs = []
    for j in hits[:limit]:
        desc = re.sub(r"<[^>]+>", " ", j.get("description", ""))
        sal = ""
        if j.get("salary_min"): sal = f"${j['salary_min']:,}\u2013${j.get('salary_max', j['salary_min']):,}"
        jobs.append({
            "id": len(jobs) + 1,
            "title": j["position"],
            "company": j.get("company", "?"),
            "location": j.get("location") or "Remote",
            "salary": sal or "not listed",
            "tags": j.get("tags", [])[:6],
            "url": j.get("url", ""),
            "snippet": " ".join(desc.split())[:400],
        })
    print(f"(pool: {len(pool)} live jobs \u2192 {len(hits)} match '{keyword}' \u2192 showing {len(jobs)})")
    return jobs

jobs = fetch_jobs(my_profile["interested_in"])
print(f"\U0001f527 Tool returned {len(jobs)} listings:\n")
for j in jobs:
    print(f"  {j['id']}. {j['title']} \u2014 {j['company']} ({j['location']}) \u00b7 {j['salary']}")
if not jobs:
    print("\U0001f615 No matches \u2014 try a broader keyword: 'marketing', 'sales', 'analyst', 'design'...")

**Those are real jobs, live on RemoteOK right now, fetched and filtered by your code.** Notice the division of labor: the *tool* fetched ~100 raw listings, *your code* narrowed them, and next the *model* will reason over them \u2014 that pipeline shape is everywhere in AI products. A chat window can't do that reliably — a tool can. This is the entire reason agents matter in business: *the model thinks, the tools touch the real world.*

## 🧠 Step 3 — The brain: rank the matches


In [ ]:
# ── BRAIN: Gemini ranks the listings against YOUR profile ─────────
from google.genai import types

prompt = f"""You are a sharp, honest career advisor for a college student.

STUDENT PROFILE:
{json.dumps(my_profile, indent=2)}

LIVE JOB LISTINGS (real, fetched just now):
{json.dumps(jobs, indent=2)}

Task:
1. Pick the TOP 3 listings for this specific student (use the id numbers).
2. For each: two sentences — why it fits, and one honest concern.
3. Flag any listing that conflicts with their dealbreakers.
4. End with one sentence of advice for this student's search overall.
Be specific — reference their actual skills. No generic fluff."""

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt,
    config=types.GenerateContentConfig(temperature=0.3),   # advisor mode: low creativity
)
print(response.text)

## ✍️ Step 4 — Act: draft the outreach

The last step of any useful agent: **produce the thing you'd actually use.**


In [ ]:
# ── Draft a short outreach note for your #1 pick ──────────────────
TOP_PICK_ID = 1   # <- change to the id of YOUR favorite from the ranking above

pick = next(j for j in jobs if j["id"] == TOP_PICK_ID)
note = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=f"""Write a 90-word application note from this student:
{json.dumps(my_profile, indent=2)}
for this job:
{json.dumps(pick, indent=2)}
Rules: specific to the role, mentions one real skill match, zero clichés
("passionate", "team player" banned), confident but honest about being a student.""",
    config=types.GenerateContentConfig(temperature=0.7),
)
print(f"📮 Draft note for: {pick['title']} @ {pick['company']}\n")
print(note.text)
print(f"\n🔗 Apply for real: {pick['url']}")

## 🎓 What you just built — and what you didn't

| You built (proto-agent) | A real agent (Session 4) |
|---|---|
| YOU decided to call the tool | The **model decides** which tool, and when |
| One pass: fetch → rank → draft | A **loop**: act, observe results, decide next step |
| One tool | A toolbox — search, calculators, APIs, databases |
| You checked the output | It checks itself (and knows when to ask you) |

That gap — *who decides* — is the entire subject of Sessions 4–8. You're now walking in the door with the right question.

**Want to go further before Session 4?**
- Change `interested_in` and re-run — the whole pipeline adapts
- Add a field to `my_profile` (e.g., `"salary_hopes"`) and see the advisor use it
- Harder: make it fetch *two* keywords and merge the lists before ranking

> ⚠️ **Ethics, one more time:** this drafts notes for jobs you'd genuinely apply to. Auto-*sending* applications at scale is spam, gets accounts banned, and — remember Session 2 — an unwatched model will confidently hallucinate your qualifications. Human review gate stays on. (That's Session 10, and it's why it exists.)

*Whiteboard, week 1: "automated job applications." Week 3: you're building it — properly.* 🚀
